# Lab 4 Student Version (Google Colab)
## Load and Augment CIFAR-10 (torchvision)

This notebook is a **student-learning version** of the attached Lab 4 guide. It keeps **every step and topic** from the lab and adds short explanations of **what you are learning** and **why it matters**.

## Lab overview
In this lab, you will work with the **CIFAR-10** dataset using PyTorch and the **torchvision** library. You will apply **data augmentation** techniques and visualize the effect of transformations on training images. This lab helps you understand why augmentation is used in training deep learning models and how to implement it in a PyTorch workflow.

### In this lab, you will
- Load the CIFAR-10 dataset using `torchvision.datasets`
- Apply data augmentation transforms using `torchvision.transforms`
- Visualize batches of original and augmented images side by side using `matplotlib`

### Estimated completion time
- **25 minutes**

### Runtime type
- Set runtime type to: **CPU**


# Task 0 (Colab setup): Imports and environment check

### What you are learning
Confirming your environment reduces setup errors and makes results more reproducible.


In [ ]:
import torch
import torchvision
print("torch version:", torch.__version__)
print("torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:
# Optional: If torchvision is missing (rare in Colab), uncomment:
# !pip -q install torch torchvision


# Task 1: Loading CIFAR-10 using `torchvision.datasets`

### What you are learning
You are learning how to download and load CIFAR-10 and verify the number of samples in the training and test sets.

### Steps (from the lab)
1. Load original CIFAR-10 dataset (no transforms).
2. Output to verify dataset loading.
3. Review the output.


In [ ]:
# 1. Load original CIFAR-10 dataset (no transforms).
import torchvision.datasets as datasets

trainset_raw = datasets.CIFAR10(root='./data', train=True, download=True)
testset_raw = datasets.CIFAR10(root='./data', train=False, download=True)

# 2. Output to verify dataset loading.
print(f"Number of training samples: {len(trainset_raw)}")
print(f"Number of test samples: {len(testset_raw)}")

# 3. Output of task 1 should confirm CIFAR-10 sizes (commonly 50,000 train / 10,000 test).


### Task 1 explanation (learning takeaways)
- `datasets.CIFAR10(...)` loads images and labels into a dataset object.
- `len(...)` is a fast way to verify loading succeeded and the dataset is complete.


# Task 2: Applying transforms for augmentation

### What you are learning
You are learning how to define augmentation + normalization transforms with `torchvision.transforms` and apply them to the training dataset.

### Steps (from the lab)
1. Import necessary libraries.
2. Define CIFAR-10 normalization stats.
3. Define class names.
4. Define training transform (augmentations + normalization).
5. Load transformed dataset.
6. Show output for four random examples.


In [ ]:
# 1. Import necessary libraries.
from torchvision import transforms
import torchvision.datasets as datasets
import random

# 2. Define CIFAR-10 normalization stats.
mean = [0.4914, 0.4822, 0.4465]
std = [0.2023, 0.1994, 0.2010]

# 3. Define class names.
classes = ['plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

# 4. Define training transform (augmentations + normalization).
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# 5. Load transformed dataset.
trainset_aug = datasets.CIFAR10(root='./data', train=True, download=False, transform=train_transform)

# 6. Show output for four random examples.
for i in range(4):
    idx = random.randint(0, len(trainset_raw) - 1)
    orig_img, label = trainset_raw[idx]
    transformed_img, _ = trainset_aug[idx]

    print(f" Example {i+1}")
    print(f"Original Image Shape: {orig_img.size}")
    print(f"Transformed Image Shape: {transformed_img.shape}")
    print(f"Corresponding Label: {classes[label]}")
    print("-" * 50)


### Task 2 explanation (learning takeaways)
- Augmentations (flip/rotation/jitter/crop) create variation for better generalization.
- `ToTensor()` converts to `(C, H, W)` format for PyTorch.
- `Normalize(mean, std)` standardizes channel values for more stable training.


# Task 3: Visualizing batched images with `matplotlib`

### What you are learning
You are learning how to visually compare original vs. augmented images in a grid and how to apply **inverse normalization** for correct display.

### Steps (from the lab)
1. Import necessary libraries.
2. Inverse normalization to display images.
3. Randomly select 5 indices from the dataset.
4. Get original and augmented images for selected indices.
5. Create 2-column × 5-row plot.
6. Add manual column titles and plot image pairs:
   - 6.1 Original image (left)
   - 6.2 Augmented image (right)
7. Review the output grid.


In [ ]:
# 1. Import necessary libraries.
import matplotlib.pyplot as plt
import numpy as np
import random
from torchvision.transforms import Normalize

# 2. Inverse normalization to display images.
inv_normalize = Normalize(
    mean=[-0.4914 / 0.2023, -0.4822 / 0.1994, -0.4465 / 0.2010],
    std=[1 / 0.2023, 1 / 0.1994, 1 / 0.2010]
)


In [ ]:
# 3. Randomly select 5 indices from the dataset.
random_indices = random.sample(range(len(trainset_raw)), 5)

# 4. Get original and augmented images for selected indices.
original_images = [trainset_raw[i][0] for i in random_indices]
labels = [trainset_raw[i][1] for i in random_indices]
augmented_images = [train_transform(img) for img in original_images]

# 5. Create 2-column x 5-row plot.
fig, axs = plt.subplots(nrows=5, ncols=2, figsize=(9, 11))

# 6. Add manual column titles.
fig.text(0.25, 0.94, "Original (with Label)", fontsize=14, ha='center')
fig.text(0.75, 0.94, "Augmented", fontsize=14, ha='center')

for i in range(5):
    # 6.1 Original image (left).
    axs[i, 0].imshow(np.asarray(original_images[i]))
    axs[i, 0].set_title(f"{classes[labels[i]]}", fontsize=10)
    axs[i, 0].axis('off')

    # 6.2 Augmented image (right).
    aug_img = inv_normalize(augmented_images[i])
    aug_img = np.transpose(aug_img.numpy(), (1, 2, 0))
    aug_img = np.clip(aug_img, 0, 1)
    axs[i, 1].imshow(aug_img)
    axs[i, 1].axis('off')

plt.subplots_adjust(top=0.92, wspace=0.1, hspace=0.4)
plt.show()


### Task 3 explanation (learning takeaways)
- The grid makes augmentation effects obvious (cropping/rotation/color changes).
- Inverse normalization is needed because normalized tensors don’t display as “normal” images.


# Lab review

1. Data augmentation improves model generalization by introducing variation in training data.  
A. True  
B. False  

2. What does `Normalize(mean, std)` do in a PyTorch transform?  
A. Convert images to grayscale  
B. Scales pixel values to a standard range  
C. Increases brightness randomly  
D. Adds Gaussian noise  

---

## STOP
You have successfully completed this lab.


<details>
<summary><strong>Optional self-check answers (click to expand)</strong></summary>

1. **A (True)**  
2. **B**

</details>
